# The Drazin Inverse

**Seong Cho** · ACME Volume 1, Lab 14

**Lab objective:** compute the Drazin inverse, a pseudoinverse that preserves certain spectral properties of a matrix, using the Schur decomposition, then use it to compute the effective resistance of a graph and perform link prediction.

<sub>Lab specification from the *Volume 1* lab manual of [Foundations of Applied Mathematics](https://foundations-of-applied-mathematics.github.io/) (BYU ACME), © Dr. J. Humpherys, used under [CC BY 3.0 US](http://creativecommons.org/licenses/by/3.0/us/). Problem statements are quoted from the spec; the notes are condensed from it. Solutions are my own.</sub>

In [1]:
import numpy as np
from scipy import linalg as la
from numpy.linalg import matrix_power

## Definition of the Drazin Inverse

The *index* of an $n \times n$ matrix $A$ is the smallest nonnegative integer $k$ for which $\mathcal{N}(A^k) = \mathcal{N}(A^{k+1})$. The *Drazin inverse* $A^D$ of $A$ is the unique $n \times n$ matrix satisfying

- $AA^D = A^DA$,
- $A^{k+1}A^D = A^k$,
- $A^DAA^D = A^D$.

If $A$ is invertible, then $k = 0$ and $A^D = A^{-1}$. If $A$ is nilpotent ($A^j = 0$ for some nonnegative integer $j$), then $A^D$ is the zero matrix.

The lab's starter code provides a helper that computes the index. Since $\mathcal{N}(A^k) \subseteq \mathcal{N}(A^{k+1})$, the null spaces are equal exactly when the ranks of $A^k$ and $A^{k+1}$ agree.

In [2]:
# Helper function for problems 1 and 2.
def index(A, tol=1e-5):
    """Compute the index of the matrix A.

    Parameters:
        A ((n,n) ndarray): An nxn matrix.

    Returns:
        k (int): The index of A.
    """

    # test for non-singularity
    if not np.allclose(la.det(A),0):
        return 0

    n = len(A)
    k = 1
    Ak = A.copy()
    while k <= n:
        r1 = np.linalg.matrix_rank(Ak)
        r2 = np.linalg.matrix_rank(np.dot(A,Ak))
        if r1 == r2:
            return k
        Ak = np.dot(A,Ak)
        k += 1

    return k

## Problem 1: Verifying a Drazin inverse

> **Problem 1.** Write a function that accepts an $n \times n$ matrix $A$, the index $k$ of $A$, and an $n \times n$ matrix $A^D$. Use the criteria described above to determine whether or not $A^D$ is the Drazin inverse of $A$. Return `True` if $A^D$ satisfies all three conditions; otherwise, return `False`.
>
> Use the following matrices as test cases for your function.
>
> $$A = \begin{bmatrix} 1 & 3 & 0 & 0 \\ 0 & 1 & 3 & 0 \\ 0 & 0 & 1 & 3 \\ 0 & 0 & 0 & 0 \end{bmatrix}, \quad A^D = \begin{bmatrix} 1 & -3 & 9 & 81 \\ 0 & 1 & -3 & -18 \\ 0 & 0 & 1 & 3 \\ 0 & 0 & 0 & 0 \end{bmatrix}, \quad k = 1$$
>
> $$B = \begin{bmatrix} 1 & 1 & 3 \\ 5 & 2 & 6 \\ -2 & -1 & -3 \end{bmatrix}, \quad B^D = \begin{bmatrix} 0 & 0 & 0 \\ 0 & 0 & 0 \\ 0 & 0 & 0 \end{bmatrix}, \quad k = 3$$
>
> (Hint: `np.allclose()` and `np.linalg.matrix_power()` may be useful).

In [3]:
def is_drazin(A, Ad, k):
    """Verify that a matrix Ad is the Drazin inverse of A.

    Parameters:
        A ((n,n) ndarray): An nxn matrix.
        Ad ((n,n) ndarray): A candidate for the Drazin inverse of A.
        k (int): The index of A.

    Returns:
        (bool) True of Ad is the Drazin inverse of A, False otherwise.
    """
    return np.allclose(A@Ad, Ad@A) and \
           np.allclose(matrix_power(A, k+1)@Ad, matrix_power(A, k)) and \
           np.allclose(Ad@A@Ad, Ad)

The spec's two test cases, with `index()` confirming the given values of $k$. $B$ is nilpotent ($B^3 = 0$), so its Drazin inverse is zero. Changing one entry of $A^D$, or using the wrong matrix, must fail; for an invertible matrix ($k = 0$) the ordinary inverse must pass.

In [4]:
A  = np.array([[1,3,0,0],[0,1,3,0],[0,0,1,3],[0,0,0,0]])
Ad = np.array([[1,-3,9,81],[0,1,-3,-18],[0,0,1,3],[0,0,0,0]])
B  = np.array([[1,1,3],[5,2,6],[-2,-1,-3]])
Bd = np.zeros((3,3))

print("index(A) =", index(A), "  index(B) =", index(B))
print("B^3 = 0:", np.allclose(matrix_power(B, 3), 0))
print(is_drazin(A, Ad, 1), is_drazin(B, Bd, 3))

Ad_wrong = Ad.copy()
Ad_wrong[0,3] = 80
assert is_drazin(A, Ad, 1) and is_drazin(B, Bd, 3)
assert not is_drazin(A, Ad_wrong, 1)
assert not is_drazin(B, np.eye(3), 3)

rng = np.random.default_rng(14)
C = rng.random((5,5))
assert index(C) == 0 and is_drazin(C, la.inv(C), 0)
print("All checks passed.")

index(A) = 1   index(B) = 3
B^3 = 0: True
True True
All checks passed.


## Computing the Drazin Inverse

The Drazin inverse is often defined theoretically in terms of the eigenprojections of a matrix, but eigenprojections are often costly or unstable to calculate, so a different method is used. Suppose the $n \times n$ matrix $A$ can be written as

$$A = S^{-1}\begin{bmatrix} M & 0 \\ 0 & N \end{bmatrix}S, \tag{15.1}$$

where $S$ is a change of basis matrix, $N$ is nilpotent, and $M$ is the restriction of $A$ onto the range of $I - P_0$, where $P_0$ is the 0-eigenprojection in the spectral decomposition. Then

$$A^D = S^{-1}\begin{bmatrix} M^{-1} & 0 \\ 0 & 0 \end{bmatrix}S. \tag{15.2}$$

The Schur decomposition of $A$ is

$$A = QTQ^{-1}, \tag{15.3}$$

where $Q$ is orthonormal and $T$ is upper triangular. Since $T$ is similar to $A$, the eigenvalues of $A$ are listed along the diagonal of $T$, so if $A$ is singular, at least one diagonal entry of $T$ is 0. The columns that contain the 0 eigenvalues of $A$ form the nilpotent matrix $N$ in (15.1). To compute $M$ and $N$, sort the Schur decomposition so that the 0 eigenvalues are listed last along the diagonal of $T$, then use (15.2).

SciPy's `la.schur()` does not sort the decomposition by eigenvalue automatically, but its `sort` keyword does:

In [5]:
# The standard Schur decomposition.
A = np.array([[0,0,2],[-3,2,6],[0,0,1]])
T,Z = la.schur(A)
T                           # The eigenvalues (2, 0, and 1) are not sorted.

array([[ 2., -3.,  6.],
       [ 0.,  0.,  2.],
       [ 0.,  0.,  1.]])

In [6]:
# Specify a sorting function to get the desired result.
f = lambda x: abs(x) > 0
T1,Z1,k = la.schur(A, sort=f)
print(T1)
print(k)                    # k is the number of columns satisfying the sort,
                            # which is the number of nonzero eigenvalues.

[[ 2.00000000e+00 -1.11022302e-16  6.70820393e+00]
 [ 0.00000000e+00  1.00000000e+00  2.00000000e+00]
 [ 0.00000000e+00  0.00000000e+00  0.00000000e+00]]
2


The spec prints `T1[0,1]` as exactly `0.`; here it comes out as roundoff of order $10^{-16}$.

Algorithm 15.1 finds the Drazin inverse from two sorted Schur decompositions. Due to possible floating point arithmetic errors, all eigenvalues smaller than a certain tolerance are considered to be 0.

```text
Algorithm 15.1
1: procedure Drazin(A, tol)
2:     (n, n) ← shape(A)
3:     Q1, S, k1 ← schur(A, |x| > tol)        ▷ Sort the Schur decomposition.
4:     Q2, T, k2 ← schur(A, |x| ≤ tol)
5:     U ← [S:,:k1 | T:,:n−k1]               ▷ Concatenate part of S and T column-wise.
6:     U⁻¹ ← inverse(U)
7:     V ← U⁻¹AU
8:     Z ← 0n×n                              ▷ The n × n zero matrix as floats, not ints.
9:     if k1 ≠ 0 then
10:        M⁻¹ ← inverse(V:k1,:k1)
11:        Z:k1,:k1 ← M⁻¹
12:    return UZU⁻¹
```

In SciPy's notation `T, Z, sdim = la.schur(A, sort=...)`, the second returned matrix (the algorithm's $S$ and $T$) is the orthonormal Schur basis, so $U$ is built from the first $k_1$ Schur vectors for the nonzero eigenvalues and the first $n - k_1$ Schur vectors for the zero eigenvalues.

## Problem 2: The Drazin inverse via the Schur decomposition

> **Problem 2.** Write a function that accepts an $n \times n$ matrix $A$ and a tolerance for rounding eigenvalues to zero. Use Algorithm 15.1 to compute the Drazin inverse $A^D$. Use your function from Problem 1 to verify your implementation.

In [7]:
def drazin_inverse(A, tol=1e-4):
    """Compute the Drazin inverse of A.

    Parameters:
        A ((n,n) ndarray): An nxn matrix.

    Returns:
       ((n,n) ndarray) The Drazin inverse of A.
    """
    # Algorithm 15.1
    n,n = A.shape
    # Sort the Schur decomposition
    f1 = lambda x: abs(x) > tol
    Q1, S, k1 = la.schur(A, sort=f1)
    f2 = lambda x: abs(x) <= tol
    Q2, T, k2 = la.schur(A, sort=f2)
    # Concatenate part of S and T column-wise
    U = np.column_stack([S[:,:k1], T[:,:n-k1]])
    U_inv = la.inv(U)
    V = U_inv@A@U
    Z = np.zeros((n,n), dtype=float)
    if k1 != 0:
        M_inv = la.inv(V[:k1,:k1])
        Z[:k1,:k1] = M_inv
    return U@Z@U_inv

Checks: the spec's $A^D$ and $B^D$ from Problem 1 are recovered, an invertible matrix gives `la.inv()`, and random singular matrices built directly in the form (15.1) (a random invertible $3 \times 3$ block $M$ and a nonzero nilpotent $2 \times 2$ block $N$, so the index is 2) give the inverse predicted by (15.2) and pass `is_drazin()`.

In [8]:
A  = np.array([[1,3,0,0],[0,1,3,0],[0,0,1,3],[0,0,0,0]])
B  = np.array([[1,1,3],[5,2,6],[-2,-1,-3]])
print(np.round(drazin_inverse(A), 8))
assert np.allclose(drazin_inverse(A), Ad) and is_drazin(A, drazin_inverse(A), index(A))
assert np.allclose(drazin_inverse(B), 0) and is_drazin(B, drazin_inverse(B), index(B))

rng = np.random.default_rng(15)
C = rng.random((5,5))
assert np.allclose(drazin_inverse(C), la.inv(C))

for trial in range(5):
    S = rng.random((5,5))
    M = rng.random((3,3)) + np.eye(3)
    N = np.array([[0., rng.random() + 1], [0., 0.]])          # Nilpotent, N^2 = 0.
    A = la.inv(S) @ la.block_diag(M, N) @ S                     # (15.1)
    AD = la.inv(S) @ la.block_diag(la.inv(M), np.zeros((2,2))) @ S   # (15.2)
    k = index(A)
    assert k == 2
    assert np.allclose(drazin_inverse(A), AD) and is_drazin(A, drazin_inverse(A), k)
print("All checks passed.")

[[  1.  -3.   9.  81.]
 [  0.   1.  -3. -18.]
 [  0.   0.   1.   3.]
 [  0.   0.   0.   0.]]
All checks passed.


> **Achtung!** Because the algorithm for the Drazin inverse requires calculation of the inverse of a matrix, it is unstable when that matrix has a high condition number. If the algorithm does not find the correct Drazin inverse, check the condition number of $V$ from Algorithm 15.1.

> **Note.** The Drazin inverse is called a pseudoinverse because $A^D = A^{-1}$ for invertible $A$, and for noninvertible $A$, $A^D$ always exists and acts similarly to an inverse. There are other matrix pseudoinverses that preserve different qualities of $A$, including the Moore-Penrose pseudoinverse $A^\dagger$, which can be thought of as the least squares approximation to $A^{-1}$.

For the singular matrix $A$ above, the Moore-Penrose pseudoinverse from `la.pinv()` is a different matrix and fails the Drazin conditions:

In [9]:
A_pinv = la.pinv(A)
print(is_drazin(A, A_pinv, index(A)), is_drazin(A, drazin_inverse(A), index(A)))

False True


## Applications of the Drazin Inverse: Effective Resistance

The *effective resistance* between two nodes in an undirected graph measures how connected those nodes are. The concept comes from circuits: a resistor limits or regulates the flow of electricity, and two points with more resistors between them have more resistance. A circuit can be represented by a graph whose nodes are the points of interest and where the number of edges connecting two nodes is the number of resistors between them. (The spec's Figure 15.1 shows such a circuit: six nodes $a$ to $f$ with a resistor on each edge.)

Electromagnetism gives rules for computing the effective resistance by hand for simple graphs, but this is infeasible for large or complicated graphs. The Drazin inverse handles any graph. Form the adjacency matrix, whose $(i,j)$th entry is the number of connections from node $i$ to node $j$, and its Laplacian $L$ (see Problem 1 of the Image Segmentation lab). Then the effective resistance from node $i$ to node $j$ is

$$R_{ij} = \begin{cases} \left(\widetilde{L}_j\right)^D_{ii} & \text{if } i \neq j \\ 0 & \text{if } i = j \end{cases}, \tag{15.4}$$

where $\widetilde{L}_j$ is the Laplacian with the $j$th row replaced by the $j$th row of the identity matrix, and $(\widetilde{L}_j)^D$ is its Drazin inverse.

The spec's Figure 15.2 gives the effective resistance between two nodes $a$ and $b$ for several simple graphs. Nodes that are farther apart have a larger effective resistance, while nodes that are nearer or better connected have a smaller one.

| Graph | $R_{ab}$ |
|---|---|
| a path $a$ – • – • – $b$ of three edges | $3$ |
| a single edge $a$ – $b$ | $1$ |
| a triangle $a$, $b$, • | $2/3$ |
| three parallel edges between $a$ and $b$ | $1/3$ |
| two parallel edges between $a$ and $b$ | $1/2$ |
| four parallel edges between $a$ and $b$ | $1/4$ |

## Problem 3: Effective resistance

> **Problem 3.** Write a function that accepts the $n \times n$ adjacency matrix of an undirected graph. Use (15.4) to compute the effective resistance from each node to every other node. Return an $n \times n$ matrix where the $(ij)$th entry is the effective resistance from node $i$ to node $j$. Keep the following in mind:
> - The resulting matrix should be symmetric.
> - The effective resistance from a node to itself is 0.
> - Consider creating the matrix column by column instead of entry by entry. Every time you compute the Drazin inverse, the whole diagonal of the matrix can be used.
>
> Test your function using the graphs and values from Figure 15.2.

The function replaces the $j$th *column* of $L$ rather than the $j$th row. Since $L$ is symmetric, this gives the transpose of the spec's $\widetilde{L}_j$, and $(M^T)^D = (M^D)^T$ has the same diagonal, so the result is the same.

In [10]:
def effective_resistance(A):
    """Compute the effective resistance for each node in a graph.

    Parameters:
        A ((n,n) ndarray): The adjacency matrix of an undirected graph.

    Returns:
        ((n,n) ndarray) The matrix where the ijth entry is the effective
        resistance from node i to node j.
    """
    n,n = A.shape
    # Compute the degree matrix
    D = np.diag(np.sum(A, axis=1))
    # Compute the Laplacian
    L = D - A
    I = np.eye(n)
    R = np.zeros((n,n))
    for j in range(n):
        Lj = L.copy()
        # Replace the jth column of L with the jth column of an identity matrix
        Lj[:,j] = I[:,j]
        # Equation 15.4
        Ld = drazin_inverse(Lj)
        R[:,j] = np.diagonal(Ld)
        R[j,j] = 0
    return R

The six graphs of Figure 15.2, with $a$ the first node and $b$ the last. Parallel edges are entries larger than 1 in the adjacency matrix.

In [11]:
graphs = {
    "path of 3 edges":    (np.array([[0,1,0,0],[1,0,1,0],[0,1,0,1],[0,0,1,0]]), 3),
    "single edge":        (np.array([[0,1],[1,0]]), 1),
    "triangle":           (np.array([[0,1,1],[1,0,1],[1,1,0]]), 2/3),
    "3 parallel edges":   (np.array([[0,3],[3,0]]), 1/3),
    "2 parallel edges":   (np.array([[0,2],[2,0]]), 1/2),
    "4 parallel edges":   (np.array([[0,4],[4,0]]), 1/4),
}
for name, (G, Rab) in graphs.items():
    R = effective_resistance(G)
    print(f"{name:18}  R_ab = {R[0,-1]:.6f}   (expected {Rab:.6f})")
    assert np.isclose(R[0,-1], Rab)
    assert np.allclose(R, R.T) and np.allclose(np.diag(R), 0)

print(effective_resistance(graphs["path of 3 edges"][0]))

path of 3 edges     R_ab = 3.000000   (expected 3.000000)
single edge         R_ab = 1.000000   (expected 1.000000)
triangle            R_ab = 0.666667   (expected 0.666667)
3 parallel edges    R_ab = 0.333333   (expected 0.333333)
2 parallel edges    R_ab = 0.500000   (expected 0.500000)
4 parallel edges    R_ab = 0.250000   (expected 0.250000)
[[0. 1. 2. 3.]
 [1. 0. 1. 2.]
 [2. 1. 0. 1.]
 [3. 2. 1. 0.]]


An independent check: the effective resistance is also given by the Moore-Penrose pseudoinverse $L^\dagger$ of the Laplacian, $R_{ij} = L^\dagger_{ii} + L^\dagger_{jj} - 2L^\dagger_{ij}$. For a random connected graph with multiple edges:

In [12]:
def resistance_pinv(A):
    Lp = la.pinv(np.diag(A.sum(axis=1)) - A)
    return np.diag(Lp)[:,None] + np.diag(Lp)[None,:] - 2*Lp

rng = np.random.default_rng(3)
G = np.triu(rng.integers(0, 3, (8,8)), 1)
G = G + G.T
assert np.allclose(effective_resistance(G), resistance_pinv(G))
print("Matches the pseudoinverse formula.")

Matches the pseudoinverse formula.


## Link Prediction

*Link prediction* is the problem of predicting the likelihood of a future association between two unconnected nodes in a graph. The canonical example is friend suggestions on Facebook: the network is a large graph with a node for each user and an edge between "friends", and Facebook aims to predict who you would like to become friends with based on who you are friends with now, as well as discover which friends you may have in real life that you have not yet connected with online. To do this, it needs a measure of how closely two users are connected.

Effective resistance is such a measure, and nodes that are closely connected at present are more likely to be connected in the future. Given an undirected graph, the next link should connect the two unconnected nodes with the least effective resistance between them.

## Problem 4: The `LinkPredictor` class

> **Problem 4.** Write a class called `LinkPredictor` for performing link prediction. Implement the `__init__()` method so that it accepts the name of a csv file containing information about a social network. Each row of the file should contain the names of two nodes which are connected by an (undirected) edge.
>
> Store each of the names of the nodes of the graph as an ordered list. Next, create the adjacency matrix for the network where the $i$th row and column of the matrix correspond to the $i$th member of the list of node names. Finally, use your function from Problem 3 to compute the effective resistance matrix. Save the list of names, the adjacency matrix, and the effective resistance matrix as attributes.

The class below also contains the Problem 5 methods, demonstrated in the next section. The names are stored in alphabetical order.

In [13]:
class LinkPredictor:
    """Predict links between nodes of a network."""

    def __init__(self, filename='social_network.csv'):
        """Create the effective resistance matrix by constructing
        an adjacency matrix.

        Parameters:
            filename (str): The name of a file containing graph data.
        """
        names = set()
        connections = list()
        # Read the file
        with open(filename, 'r') as myfile:
            for line in myfile.readlines():
                con = line.strip().split(',')
                connections.append(con)
                names.add(con[0])
                names.add(con[1])
        self.names = sorted(list(names))
        n = len(self.names)
        self.n = n
        # Compute the adjacency matrix
        A = np.zeros((n,n))
        for con in connections:
            i = self.names.index(con[0])
            j = self.names.index(con[1])
            A[i,j], A[j,i] = 1,1
        self.adj = A
        # Compute the effective resistance matrix
        self.erm = effective_resistance(A)

    def predict_link(self, node=None):
        """Predict the next link, either for the whole graph or for a
        particular node.

        Parameters:
            node (str): The name of a node in the network.

        Returns:
            node1, node2 (str): The names of the next nodes to be linked.
                Returned if node is None.
            node1 (str): The name of the next node to be linked to 'node'.
                Returned if node is not None.

        Raises:
            ValueError: If node is not in the graph.
        """
        # Replace the values of the resistance matrix with 0s where
        # connections already exist
        erm = self.erm * (np.ones((self.n,self.n)) - self.adj)
        if node == None:
            minval = np.min(erm[erm>0])
            loc = np.where(erm==minval)
            return self.names[loc[0][0]], self.names[loc[1][0]]
        elif node in self.names:
            i = self.names.index(node)
            minval = np.min(erm[i][erm[i]>0])
            loc = np.where(erm==minval)
            if self.names[loc[1][0]] == node:
                return self.names[loc[0][0]]
            else:
                return self.names[loc[1][0]]
        else:
            raise ValueError("Node is not in the network")

    def add_link(self, node1, node2):
        """Add a link to the graph between node 1 and node 2 by updating the
        adjacency matrix and the effective resistance matrix.

        Parameters:
            node1 (str): The name of a node in the network.
            node2 (str): The name of a node in the network.

        Raises:
            ValueError: If either node1 or node2 is not in the graph.
        """
        if node1 not in self.names or node2 not in self.names:
            raise ValueError("Node is not in the network")
        else:
            i = self.names.index(node1)
            j = self.names.index(node2)
            self.adj[i,j] += 1
            self.adj[j,i] += 1
            self.erm = effective_resistance(self.adj)

`social_network.csv` has 78 edges. The spec's Figure 15.3 (a drawing of the network, adapted from Wayne W. Zachary's karate club data, see https://en.wikipedia.org/wiki/Zachary%27s_karate_club) numbers 34 people, but two of them are both called Jake (numbers 12 and 25), so the file has 33 distinct names and the two Jakes become one node.

In [14]:
LP = LinkPredictor(filename='social_network.csv')
print(len(LP.names), "names:", LP.names)
print("edges:", int(LP.adj.sum() // 2))
assert np.allclose(LP.adj, LP.adj.T) and np.allclose(np.diag(LP.adj), 0)
assert np.allclose(LP.erm, effective_resistance(LP.adj))
assert np.allclose(LP.erm, resistance_pinv(LP.adj))

33 names: ['Abigail', 'Alan', 'Alexander', 'Anna', 'Brandon', 'Carol', 'Charles', 'Christopher', 'Colin', 'Connor', 'Emily', 'Eric', 'Evan', 'Jake', 'Jane', 'John', 'Madeleine', 'Mary', 'Max', 'Melanie', 'Oliver', 'Paul', 'Penelope', 'Piers', 'Ruth', 'Sally', 'Sonia', 'Stephanie', 'Stephen', 'Theresa', 'Thomas', 'Tracey', 'Trevor']
edges: 78


## Problem 5: Predicting and adding links

> **Problem 5.** Implement the following methods in the `LinkPredictor` class:
> 1. `predict_link()`: Accept a parameter `node` which is either `None` or a string representing a node in the network. If `node` is `None`, return a tuple with the names of the nodes between which the next link should occur. However, if `node` is a string, return the name of the node which should be connected to `node` next out of all other nodes in the network. If `node` is not in the network, raise a `ValueError`. Take the following into consideration:
>     - (a) You want to find the two nodes which have the smallest effective resistance between them which are not yet connected. Use information from the adjacency matrix to zero out all entries of the effective resistance matrix that represent connected nodes. The "`*`" operator multiplies arrays component-wise, which may be helpful.
>     - (b) Find the next link by finding the minimum value of the array that is nonzero. Your array may be the whole matrix or just a column if you are only considering links for a certain node. This can be accomplished by passing `np.min()` a masked version of your matrix to exclude entries that are 0.
>     - (c) NumPy's `np.where()` is useful for finding the minimum value in an array:
>
>        ```python
>        >>> A = np.random.randint(-9,9,(3,3))
>        >>> A
>        array([[ 6, -8, -9],
>               [-2,  1, -1],
>               [ 4,  0, -3]])
>
>        # Find the minimum value in the array.
>        >>> minval = np.min(A)
>        >>> minval
>        -9
>
>        # Find the location of the minimum value.
>        >>> loc = np.where(A==minval)
>        >>> loc
>        (array([0], dtype=int64), array([2], dtype=int64))
>        ```
> 2. `add_link()`: Take as input two names of nodes, and add a link between them. If either name is not in the network, raise a `ValueError`. Add the link by updating the adjacency matrix and the effective resistance matrix.
>
> Figure 15.3 visualizes the data in `social_network.csv`. Use this graph to verify that your class is suggesting plausible new links. You should observe the following:
> - In the entire network, Emily and Oliver are most likely to become friends next.
> - Melanie is predicted to become friends with Carol next.
> - Alan is expected to become friends with Sonia, then with Piers, and then with Abigail.

The same `np.where()` steps, runnable (current NumPy prints the index arrays without the `dtype=int64` tag):

In [15]:
np.random.seed(5)
A = np.random.randint(-9,9,(3,3))
print(A)
minval = np.min(A)                  # Find the minimum value in the array.
print(minval)
loc = np.where(A==minval)           # Find the location of the minimum value.
loc

[[-6  5  6]
 [-3  7  0]
 [-1 -5 -2]]
-6


(array([0]), array([0]))

The spec's expected predictions:

In [16]:
LP = LinkPredictor(filename='social_network.csv')
print(LP.predict_link())
print(LP.predict_link(node="Melanie"))

alan = []
for _ in range(3):
    alan.append(LP.predict_link(node="Alan"))
    LP.add_link("Alan", alan[-1])
print("Alan:", alan)

assert set(LinkPredictor().predict_link()) == {"Emily", "Oliver"}
assert LinkPredictor().predict_link(node="Melanie") == "Carol"
assert alan == ["Sonia", "Piers", "Abigail"]

('Oliver', 'Emily')
Carol
Alan: ['Sonia', 'Piers', 'Abigail']


In [17]:
for bad_call in [lambda: LP.predict_link(node="Nobody"),
                 lambda: LP.add_link("Alan", "Nobody"),
                 lambda: LP.add_link("Nobody", "Alan")]:
    try:
        bad_call()
    except ValueError as e:
        print("ValueError:", e)

ValueError: Node is not in the network
ValueError: Node is not in the network
ValueError: Node is not in the network
